# RAG real-estate advice chatbot (learning notebook)

Run the cells one by one from the top with **Shift + Enter**.

Keep this notebook in the same folder as the 3 data files: `listings.csv`, `projects.md`, `buying_faq.md`.

## Step 1: Install the libraries

Only needed once. After installing you may have to restart the kernel (Kernel > Restart).

In [ ]:
%pip install -q anthropic chromadb sentence-transformers

In [ ]:
import csv
import os
import re
from getpass import getpass

import anthropic
import chromadb
from chromadb.utils import embedding_functions

MODEL = "claude-sonnet-5-5"  # switch to another model if you want it cheaper
CHUNKS_PER_QUESTION = 3      # number of document chunks given to Claude per question

## Step 2: Split the documents into chunks

- `.md` files: every section starting with `##` is one chunk.
- `.csv` file: every listing row is one chunk, rewritten as a sentence.

In [ ]:
def read_markdown(path):
    """Every section starting with '## ' in a .md file becomes one chunk."""
    with open(path, encoding="utf-8") as f:
        content = f.read()

    chunks = []
    # Split on lines starting with "## " and drop whatever comes before the first section
    for part in re.split(r"^## ", content, flags=re.MULTILINE)[1:]:
        heading, _, body = part.partition("\n")
        chunks.append({
            "text": f"{heading.strip()}\n{body.strip()}",
            "source": f"{path} > {heading.strip()}",
        })
    return chunks


def read_csv(path):
    """Every listing row becomes one chunk, rewritten as a (Vietnamese) sentence."""
    type_labels = {"apartment": "Căn hộ", "townhouse": "Nhà phố", "land": "Đất nền"}

    chunks = []
    with open(path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            project = f", thuộc dự án {row['project']}" if row["project"] else ""
            text = (
                f"Tin đăng {row['id']}: {row['title']}. "
                f"Loại: {type_labels.get(row['type'], row['type'])}{project}. "
                f"Vị trí: {row['district']}, {row['city']}. "
                f"Giá: {row['price_billion_vnd']} tỷ đồng. "
                f"Diện tích: {row['area_m2']} m2. "
                f"{row['bedrooms']} phòng ngủ, {row['bathrooms']} phòng tắm. "
                f"Pháp lý: {row['legal_status']}. "
                f"Ngày đăng: {row['posted_date']}. "
                f"Mô tả: {row['description']}"
            )
            chunks.append({"text": text, "source": f"{path} > {row['id']}"})
    return chunks


chunks = (
    read_markdown("projects.md")
    + read_markdown("buying_faq.md")
    + read_csv("listings.csv")
)
print(f"{len(chunks)} chunks in total")

See the list of chunks:

In [ ]:
for i, c in enumerate(chunks):
    print(f"{i:>2}  {len(c['text']):>4} chars  {c['source']}")

See the full text of any chunk (change the number in square brackets to see another one):

In [ ]:
print(chunks[0]["text"])

## Step 3: Create embeddings and store them in Chroma

The first run downloads the embedding model (about 500MB), so it takes a while.

In [ ]:
# Multilingual embedding model, runs locally for free
embedding = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

client = chromadb.Client()  # kept in memory, lost when the kernel stops

# Delete the old collection if any, so this cell can be re-run without a name clash
try:
    client.delete_collection("real_estate")
except Exception:
    pass

store = client.create_collection("real_estate", embedding_function=embedding)

# Chroma calls the embedding model for each chunk as it is added
store.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
)
print(f"The store has {store.count()} chunks")

See what a vector looks like:

In [ ]:
vector = embedding(["thủ tục sang tên sổ đỏ"])[0]
print(f"The vector has {len(vector)} numbers. The first 8:")
print([round(float(x), 3) for x in vector[:8]])

In [ ]:
print(chunks[0]["text"])

## Step 4: Search

This cell **does not use Claude yet**. Results are printed with their *distance*: the smaller the number, the closer the chunk's meaning is to the question.

Test thoroughly here first: if the right chunk does not show up, Claude cannot answer correctly either.

In [ ]:
def search(question, k=CHUNKS_PER_QUESTION):
    """Return the k chunks whose meaning is closest to the question."""
    results = store.query(query_texts=[question], n_results=k)
    return [
        {"text": text, "source": meta["source"], "distance": dist}
        for text, meta, dist in zip(
            results["documents"][0], results["metadatas"][0], results["distances"][0]
        )
    ]


def show_search(question, k=CHUNKS_PER_QUESTION):
    print(f"Question: {question}")
    for c in search(question, k):
        print(f"  {c['distance']:.3f}  {c['source']}")
    print()


show_search("Mua nhà phải đóng những loại thuế phí gì?")
show_search("Dự án nào gần metro?")
show_search("An Lạc Garden thanh toán thế nào?")

## Step 5: Connect Claude

The cell below asks for your API key if the `ANTHROPIC_API_KEY` environment variable is not set. The key is typed hidden and is not saved in the notebook.

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Claude API key: ")

claude = anthropic.Anthropic()

In [ ]:
SYSTEM_PROMPT = """You are a real-estate advice assistant.
Answer only from the document chunks provided inside <document> tags.
If the documents do not contain the answer, say clearly that you do not have that information; do not guess.
At the end of the answer, cite the sources you used as [Nguồn: ...].
Answer briefly, in Vietnamese."""


def ask(question, k=CHUNKS_PER_QUESTION):
    """Find the relevant chunks, send them to Claude, print both."""
    found = search(question, k)

    print(f"Question: {question}\n")
    print("Chunks found:")
    for c in found:
        print(f"  {c['distance']:.3f}  {c['source']}")

    documents = "\n\n".join(
        f"<document source=\"{c['source']}\">\n{c['text']}\n</document>" for c in found
    )
    response = claude.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"{documents}\n\nQuestion: {question}"}],
    )
    print(f"\nChatbot: {response.content[0].text}")

## Step 6: Try 5 test questions

One question per cell so the results are easy to compare.

In [ ]:
ask("Mua nhà phải đóng những loại thuế phí gì?")

In [ ]:
ask("Dự án nào gần metro?")

In [ ]:
ask("An Lạc Garden thanh toán thế nào?")

In [ ]:
ask("Căn 2 phòng ngủ dưới 3 tỷ ở Quận 7?")

In [ ]:
ask("Giá nhà ở Hà Nội thế nào?")

Ask your own question:

In [ ]:
ask("Write your question here")